# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — the same cohort, gate and model as ML-08

Rebuilt here rather than imported, so this notebook stands alone and its numbers can be checked
against ML-08's without trusting a shared file. The printed totals must match ML-08 exactly; if they
do not, one of the two notebooks has drifted and nothing below is comparable.

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy scipy scikit-learn matplotlib python-dotenv

import os
import sys
import duckdb
import numpy as np
import pandas as pd
import sklearn
from huggingface_hub import hf_hub_download

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

SEED = 8
print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | "
      f"numpy {np.__version__} | scikit-learn {sklearn.__version__}")

token = os.environ.get("HF_TOKEN")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face token (read): ")

REPO = "FlyRank/internship-warehouse"
MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]
daily_files = [hf_hub_download(repo_id=REPO, repo_type="dataset",
               filename=f"fact_content_daily_performance/month={m}/data_0.parquet", token=token)
               for m in MONTHS]
dim_file = hf_hub_download(repo_id=REPO, repo_type="dataset",
                           filename="dim_content.parquet", token=token)
con = duckdb.connect()
REL = "read_parquet([" + ", ".join(f"'{f}'" for f in daily_files) + "])"
D1 = "2026-03-31"

q = f"""
WITH prior AS (
  SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
    SUM(gsc_impressions) AS impr_90d,
    SUM(gsc_clicks) AS clicks_90d,
    COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions,
    SUM(gsc_sum_position) AS sum_position_90d,
    SUM(ga4_sessions) AS ga4_sessions_90d,
    SUM(ga4_engaged_sessions) AS ga4_engaged_90d,
    SUM(ga4_pageviews) AS ga4_pageviews_90d,
    SUM(scroll_events) AS scroll_events_90d,
    BOOL_OR(ga4_data_available) AS ga4_available,
    SUM(gsc_impressions) FILTER (
        WHERE report_date < DATE '{D1}' - INTERVAL 30 DAY) AS older60_impr,
    SUM(gsc_impressions) FILTER (
        WHERE report_date >= DATE '{D1}' - INTERVAL 30 DAY) AS recent30_impr,
    SUM(gsc_impressions) FILTER (
        WHERE report_date >= DATE '{D1}' - INTERVAL 60 DAY
          AND report_date <  DATE '{D1}' - INTERVAL 30 DAY) AS base30_impr
  FROM {REL}
  WHERE report_date >= DATE '{D1}' - INTERVAL 90 DAY AND report_date < DATE '{D1}'
  GROUP BY content_hash_id HAVING SUM(gsc_impressions) > 0),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS future_impr FROM {REL}
  WHERE report_date >= DATE '{D1}' AND report_date < DATE '{D1}' + INTERVAL 30 DAY
  GROUP BY content_hash_id)
SELECT p.*, COALESCE(f.future_impr, 0) AS future_impr
FROM prior p LEFT JOIN future f USING (content_hash_id)
ORDER BY p.content_hash_id"""

df = con.sql(q).df()
dim = con.sql(f"""SELECT content_hash_id, content_created_date, content_updated_date,
                         content_type, main_intent, competition_level,
                         search_volume, cpc, competition, backlinks,
                         keyword_char_count, keyword_token_count,
                         url_char_count, category_count,
                         keyword_created_date, last_optimized_date,
                         optimization_eligible_date
                  FROM read_parquet('{dim_file}')""").df()
df = df.merge(dim, on="content_hash_id", how="left")
for c in ["older60_impr", "recent30_impr", "base30_impr"]:
    df[c] = df[c].fillna(0)

DECISION = pd.Timestamp(D1)
df["baseline_daily"] = df["older60_impr"] / 60
df["recent_daily"] = df["recent30_impr"] / 30
df["future_daily"] = df["future_impr"] / 30
df["target"] = np.arcsinh(df["future_daily"]) - np.arcsinh(df["baseline_daily"])
df["declined"] = df["target"] < 0
df["avg_position"] = df["sum_position_90d"] / df["impr_90d"].replace(0, np.nan) + 1
df["ctr"] = df["clicks_90d"] / df["impr_90d"].replace(0, np.nan) * 100
df["log_impr_90d"] = np.log1p(df["impr_90d"])
df["engaged_rate"] = df["ga4_engaged_90d"] / df["ga4_sessions_90d"].replace(0, np.nan)
df["pages_per_session"] = df["ga4_pageviews_90d"] / df["ga4_sessions_90d"].replace(0, np.nan)
df["scroll_per_session"] = df["scroll_events_90d"] / df["ga4_sessions_90d"].replace(0, np.nan)
df["log_ga4_sessions"] = np.log1p(df["ga4_sessions_90d"].fillna(0))
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["days_with_impressions"] = df["days_with_impressions"].fillna(0)
df["content_age_days"] = (DECISION - pd.to_datetime(df["content_created_date"])).dt.days
df["slip"] = np.where(df["baseline_daily"] > 0,
                      (df["baseline_daily"] - df["recent_daily"]) / df["baseline_daily"], np.nan)
df["peak_ratio"] = np.where(df["impr_90d"] > 0,
                            df["recent_daily"] / (df["impr_90d"] / 90), np.nan)
df["prior_trend"] = np.where(df["base30_impr"] > 0,
                             (df["recent30_impr"] - df["base30_impr"]) / df["base30_impr"], np.nan)

# --- the frozen ML-07 gate, verbatim
MIN_AGE_DAYS = 180
client_median_impr = df.groupby("client_hash_id")["impr_90d"].transform("median")
gate = ((df["baseline_daily"] > 0)
        & (df["content_age_days"] >= MIN_AGE_DAYS)
        & (df["impr_90d"] >= client_median_impr)
        & (df["slip"].fillna(0) <= 0.5))
df["in_gate"] = gate
pool = df[gate].copy()

print(f"cohort {len(df):,} pages | {df['client_hash_id'].nunique()} clients")
print(f"gated pool {len(pool):,} pages | {pool['client_hash_id'].nunique()} clients")
print(f"cohort decline rate {df['declined'].mean():.4f} | pool {pool['declined'].mean():.4f}")
print()
print("ML-07 reported: cohort 202,073 / 53 clients, pool 46,061, "
      "cohort rate 0.4228, pool rate 0.5128")

from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

SAFE = ["content_age_days", "impr_90d", "avg_position"]
FULL = SAFE + ["prior_trend", "peak_ratio"]
K = 100
ev = pool.dropna(subset=FULL).copy()


def rank_agreement(pred, actual, ascending):
    rho = spearmanr(pred, actual).statistic
    return rho if ascending else -rho


print()
print(f"evaluation pool {len(ev):,} pages | {ev['client_hash_id'].nunique()} clients "
      f"| decline rate {ev['declined'].mean():.4f}")
print("ML-08 reported: 45,095 pages, 30 clients, 0.5131")

LEAN = [f for f in FULL if f != "content_age_days"]


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python 3.13.14 | pandas 3.0.1 | numpy 2.4.4 | scikit-learn 1.9.0


cohort 202,073 pages | 53 clients
gated pool 46,061 pages | 30 clients
cohort decline rate 0.4228 | pool 0.5128

ML-07 reported: cohort 202,073 / 53 clients, pool 46,061, cohort rate 0.4228, pool rate 0.5128



evaluation pool 45,095 pages | 30 clients | decline rate 0.5131
ML-08 reported: 45,095 pages, 30 clients, 0.5131


## 1. Two findings from the FlyRank paper, and the questions I would ask

`docs/flyrank-seo-research-march-2026.pdf` is a careful piece of work. It leads with direct
comparisons, keeps negative results visible, and labels its ML section as exploratory. Both questions
below are offered in that spirit — each is a check that could be run, not a claim that the finding is
wrong.

### Finding #4 — "The Freshness Multiplier" (7.88:1 growth ratio at 31–90 days)

**Where the label comes from.** Pages are bucketed by *freshness* — days since last content update —
and each bucket gets a growth-to-decline ratio. Freshness comes from `content_updated_date`.

**The question.** In the warehouse we were given, `dim_content` is an **export-time snapshot**:
`content_updated_date` records the page's state when the table was exported, not its state on any
earlier date. ML-06 Finding 1 measured this — the field sits *after* our decision point for **77.3%**
of pages at D1, with a single bulk date accounting for a large share of the portfolio.

If the production export shares that behaviour, a page refreshed *last week* appears in the "0–30 day"
freshness bucket even when the comparison window it is being scored over ended months ago. The
freshest bucket would then be enriched with pages that were updated **because** someone already
noticed them growing or declining — which is the direction of causation the finding reads the other
way.

The paper's own caution about the `361+` bucket (283:1 on one declining page) shows the team already
watches for this class of problem. The cell below runs the check on our warehouse so the question
comes with a method attached rather than a worry.

### ML Appendix — "What Predicts Growth?" (logistic regression, 71% holdout accuracy)

**Where the label comes from.** `trend_direction`, itself derived from 30-day-versus-previous-30-day
impression change.

**The question is one number.** The paper's own Finding #1 reports the split as **74.8K** growing
against **45.6K** declining. Always predicting "growing" therefore scores about **62%**. Reported next
to it, 71% reads as roughly **9 points of skill** rather than 71 — a materially different claim, and
one the paper's evidence standard would support making explicitly.

This is the check this project adopted after making the same mistake: ML-08 reported per-client
precision from 0.458 to 0.970 as a model weakness, and ML-09 §2a found the spread was base rates and
queue capacity. A base rate beside every score is cheap and catches exactly this.

In [2]:
# The freshness check, run on our warehouse so the question comes with a method.
upd = pd.to_datetime(df["content_updated_date"], errors="coerce")
DEC = pd.Timestamp(D1)
df["days_since_update_snapshot"] = (DEC - upd).dt.days
verifiable = upd.notna() & (upd <= DEC)

print(f"pages with content_updated_date present: {int(upd.notna().sum()):,}")
print(f"  of those, dated AFTER the decision point: "
      f"{int((upd > DEC).sum()):,} ({(upd > DEC).sum() / max(upd.notna().sum(), 1):.1%})")
print(f"  verifiably before it: {int(verifiable.sum()):,}")
print()

BANDS = [(-10**9, 30, "0-30"), (30, 90, "31-90"), (90, 180, "91-180"),
         (180, 360, "181-360"), (360, 10**9, "361+")]


def growth_ratio(frame, label):
    rows = []
    for lo, hi, name in BANDS:
        b = frame[(frame["days_since_update_snapshot"] > lo)
                  & (frame["days_since_update_snapshot"] <= hi)]
        if len(b) < 50:
            rows.append({"freshness": name, "n": len(b), "ratio": np.nan}); continue
        grow = int((b["target"] > 0).sum()); dec = int((b["target"] < 0).sum())
        rows.append({"freshness": name, "n": len(b),
                     "growing": grow, "declining": dec,
                     "ratio": round(grow / dec, 2) if dec else np.nan})
    out = pd.DataFrame(rows)
    print(label)
    print(out.to_string(index=False))
    print()
    return out


naive = growth_ratio(df[df["days_since_update_snapshot"].notna()],
                     "AS THE SNAPSHOT REPORTS IT (every page, updated date taken at face value)")
clean = growth_ratio(df[verifiable], "RESTRICTED to pages whose update is verifiably pre-decision")

print("The two tables answer different questions. The first buckets pages by a date")
print("that may postdate the outcome window; the second uses only updates that had")
print("already happened when the decision was made. Where the ratios differ, the")
print("freshness signal is partly recording what was done to the page afterwards.")

# How much of each bucket survives the restriction? The freshest bucket is the
# one that can contain negative "days since update", so it is the one to watch.
merged = naive.merge(clean, on="freshness", suffixes=("_naive", "_verifiable"))
merged["kept_pct"] = (merged["n_verifiable"] / merged["n_naive"] * 100).round(1)
merged["shrinkage_pct"] = (100 - merged["kept_pct"]).round(1)
print()
print("how much of each bucket is post-decision:")
print(merged[["freshness", "n_naive", "n_verifiable", "kept_pct", "shrinkage_pct",
              "ratio_naive", "ratio_verifiable"]].to_string(index=False))

pages with content_updated_date present: 202,073
  of those, dated AFTER the decision point: 161,525 (79.9%)
  verifiably before it: 40,548



AS THE SNAPSHOT REPORTS IT (every page, updated date taken at face value)
freshness      n  growing  declining  ratio
     0-30 162458  95548.0    59685.0   1.60
    31-90  33863  12632.0    21028.0   0.60
   91-180   1957    636.0     1114.0   0.57
  181-360   3795    116.0     3611.0   0.03
     361+      0      NaN        NaN    NaN

RESTRICTED to pages whose update is verifiably pre-decision
freshness     n  growing  declining  ratio
     0-30   933    172.0      744.0   0.23
    31-90 33863  12632.0    21028.0   0.60
   91-180  1957    636.0     1114.0   0.57
  181-360  3795    116.0     3611.0   0.03
     361+     0      NaN        NaN    NaN

The two tables answer different questions. The first buckets pages by a date
that may postdate the outcome window; the second uses only updates that had
already happened when the decision was made. Where the ratios differ, the
freshness signal is partly recording what was done to the page afterwards.

how much of each bucket is post-decisio

**Result: in this warehouse, the freshness signal reverses when the check is applied.**

**161,525 of 202,073 pages — 79.9% — carry a `content_updated_date` that falls after the decision
point.** ML-06 Finding 1 measured 77.3% on its narrower cohort; on the full cohort it is worse.

| freshness bucket | as the snapshot reports it | restricted to verifiable updates |
|---|---|---|
| **0–30 days** | 162,458 pages, ratio **1.60** | **933** pages, ratio **0.23** |
| 31–90 | 33,863, ratio 0.60 | 33,863, ratio 0.60 |
| 91–180 | 1,957, ratio 0.57 | 1,957, ratio 0.57 |
| 181–360 | 3,795, ratio 0.03 | 3,795, ratio 0.03 |

**Taken at face value the freshest bucket grows at 1.60:1. Restricted to updates that had actually
happened by the decision point, the same bucket declines at 0.23:1** — the direction reverses, and
the bucket loses **99.4%** of its pages.

**Why the freshest bucket specifically.** A naive `days_since_update` is `decision_date − updated_date`,
which goes *negative* when the update postdates the decision. Negative values land in the lowest
bucket, so every post-decision update is silently filed as "just refreshed". The other buckets are
unchanged precisely because they cannot contain negatives.

**What this does and does not say.** It says the check is cheap and that this failure mode is real in
the data FlyRank gave interns. It does **not** say Finding #4 is wrong — the paper works from a
different export, its 7.88:1 sits in the 31–90 bucket rather than 0–30, and that bucket is stable
here. The paper's own note about the `361+` bucket shows the team already looks for exactly this.

**The question worth asking, then, is narrow:** does the production export record
`content_updated_date` as of the reporting window, or as of export time? If the latter, the 0–30
freshness bucket is measuring interventions rather than predicting them, and the fix is a timestamp
rather than an analysis change.

## 2a. Does one model serve every client, or is the population two populations?

ML-08's error analysis found per-client `P@100` spanning **0.458 to 0.970** on a single held-out
split, with the worst client taking 72 picks — large enough that it is not a sample-size artefact.
That is a validation question, and it is the one ML-09 should open with.

**What ML-08 already ruled out.** It tested whether *data-rich clients gain from features the others
lack*: inside the 19 of 30 clients holding keyword data, adding `search_volume`, `cpc` and
`competition` cost **−0.0072** spearman. So segmentation-by-available-columns does not help.

**What it never asked.** Whether clients differ in the *relationship itself* — the same features
predicting differently from one client to the next. If they do, a shared model is averaging over
populations that should be modelled apart, and per-client variance is structural rather than noise.

**Find the segmentation variable, do not assume it.** Score every client across all ten splits, then
ask what client-level property predicts where the model fails. If nothing does, the variance is noise
and one model is correct. If something does, that property is the segmentation candidate — and only
then is a segmented model worth building.

In [3]:
# Every client appears in the test set of roughly two splits out of ten.
# Pool those appearances so each client gets a performance record.
per_client_rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te].copy()
    sc_ = StandardScaler().fit(train[FULL])
    m = Ridge(alpha=1.0).fit(sc_.transform(train[FULL]), train["target"])
    test["pred"] = m.predict(sc_.transform(test[FULL]))
    for cid, g in test.groupby("client_hash_id"):
        k = min(K, len(g))
        top = g.nsmallest(k, "pred")
        rnd = g.sample(k, random_state=seed)
        per_client_rows.append({
            "client_hash_id": cid, "seed": seed, "pages": len(g),
            "picks": k,
            "p_at_k": top["declined"].mean(),
            "p_random": rnd["declined"].mean(),
            "base_rate": g["declined"].mean(),
            "spearman": rank_agreement(g["pred"].values, g["target"].values, True)
                        if len(g) > 20 else np.nan,
        })

pc = pd.DataFrame(per_client_rows)
prof = (pc.groupby("client_hash_id")
        .agg(appearances=("seed", "nunique"), pages=("pages", "mean"),
             picks=("picks", "mean"), p_at_k=("p_at_k", "mean"),
             p_random=("p_random", "mean"), base_rate=("base_rate", "mean"),
             spearman=("spearman", "mean")))
prof["lift"] = prof["p_at_k"] - prof["p_random"]
prof = prof.sort_values("lift")

print(f"{len(prof)} clients scored, {prof['appearances'].sum()} client-appearances "
      f"across 10 splits")
print()
print(prof.round(4).to_string())

27 clients scored, 60 client-appearances across 10 splits

                         appearances   pages  picks  p_at_k  p_random  base_rate  spearman    lift
client_hash_id                                                                                    
client_08d2847f24cf89c1            2    56.0   56.0  0.5714    0.5714     0.5714    0.4563  0.0000
client_0e1acc6cd57b0eba            2     6.0    6.0  0.5000    0.5000     0.5000       NaN  0.0000
client_599043c0ff13edea            3     5.0    5.0  0.6000    0.6000     0.6000       NaN  0.0000
client_8dbf3abdf07569e0            3    13.0   13.0  0.3846    0.3846     0.3846       NaN  0.0000
client_764ae36a94e30a25            3     1.0    1.0  0.0000    0.0000     0.0000       NaN  0.0000
client_795153d5b7850ccf            3    75.0   75.0  0.7733    0.7733     0.7733    0.3438  0.0000
client_59256b0571e0c970            3     2.0    2.0  0.5000    0.5000     0.5000       NaN  0.0000
client_a1203ffecad62470            3     1.0    1.

In [4]:
# Three candidate explanations for the per-client spread, tested in order of
# how boring they are. A structural cause has to be ruled out before
# "clients differ" is allowed to stand.
prof2 = prof.copy()
prof2["rankable"] = prof2["pages"] > K
prof2["br_distance"] = (prof2["base_rate"] - 0.5).abs()

zero = prof2[prof2["lift"] <= 0.0001]
print(f"clients with exactly zero lift: {len(zero)} of {len(prof2)}")
print(f"  of those, how many have pages <= K={K}: {(zero['pages'] <= K).sum()}")
print(f"  their page counts: {sorted(zero['pages'].astype(int))}")
print()
print("When a client has fewer pages than the queue has slots, the queue takes")
print("every page and no ranking happens. Lift is zero by construction, not by")
print("failure -- and precision equals that client's own base rate exactly.")
print()

rank_ok = prof2[prof2["rankable"]]
print(f"among the {len(rank_ok)} clients with more pages than slots:")
print(f"  lift range {rank_ok['lift'].min():.4f} - {rank_ok['lift'].max():.4f}")
print(f"  Spearman(lift, |base_rate - 0.5|) = "
      f"{rank_ok['lift'].corr(rank_ok['br_distance'], method='spearman'):+.4f}")
print(f"  Spearman(lift, pages)             = "
      f"{rank_ok['lift'].corr(rank_ok['pages'], method='spearman'):+.4f}")
print()
print("A base rate far from 0.5 caps how much any ranking can add: at 0.91 a")
print("random queue already scores 0.91 and only 0.09 of headroom exists.")
print()

# The capacity-free, base-rate-free measure: does the model order pages
# correctly INSIDE each client, regardless of how many slots the queue has?
sp = prof2["spearman"].dropna()
print(f"per-client rank agreement, the measure neither capacity nor base rate distorts:")
print(f"  clients with enough pages to compute it: {len(sp)} of {len(prof2)}")
print(f"  range {sp.min():.4f} - {sp.max():.4f} | median {sp.median():.4f}")
print(f"  clients with NEGATIVE rank agreement: {(sp < 0).sum()}")

clients with exactly zero lift: 12 of 27
  of those, how many have pages <= K=100: 12
  their page counts: [1, 1, 2, 5, 6, 13, 22, 56, 72, 75, 78, 100]

When a client has fewer pages than the queue has slots, the queue takes
every page and no ranking happens. Lift is zero by construction, not by
failure -- and precision equals that client's own base rate exactly.

among the 15 clients with more pages than slots:
  lift range 0.0100 - 0.5700
  Spearman(lift, |base_rate - 0.5|) = -0.4714
  Spearman(lift, pages)             = +0.5321

A base rate far from 0.5 caps how much any ranking can add: at 0.91 a
random queue already scores 0.91 and only 0.09 of headroom exists.

per-client rank agreement, the measure neither capacity nor base rate distorts:
  clients with enough pages to compute it: 21 of 27
  range 0.3438 - 0.7852 | median 0.4814
  clients with NEGATIVE rank agreement: 0


**Verdict: the per-client spread is capacity and base rates. The model works for every client, and
ML-08's "worst client" finding is withdrawn.**

**Twelve of 27 clients have exactly zero lift, and all twelve have fewer pages than the queue has
slots** — their page counts are 1, 1, 2, 5, 6, 13, 22, 56, 72, 75, 78 and 100. With `pages <= K = 100` the queue takes the whole client, no ranking occurs, and precision
equals that client's own base rate by arithmetic.

**That includes ML-08's worst client.** `client_d211cb07b9059bab` scored **0.458** — and its base rate
is **0.4583**. It has **72 pages** against 100 slots. ML-08 reported it as a model weakness and argued
it was not a sample-size artefact because 72 picks is a lot. **That reasoning was wrong.** It is not a
sample-size artefact; it is a *capacity* artefact. The model never ranked anything for that client,
because there was nothing to rank.

**Among the 15 clients that can actually be ranked, lift tracks the base rate rather than client
character** — `Spearman(lift, |base_rate - 0.5|)` is **-0.4714**, and `Spearman(lift, pages)` is
**+0.5321**. Neither alone explains it; together with the twelve structural zeros they leave little
for client character to account for. A client whose pages decline at 0.9138 leaves only 0.09 of headroom above random, and
scores 0.0833. A client at 0.1662 leaves plenty, and scores 0.5567.

**The measure that survives both distortions is per-client rank agreement.** It does not care how many
slots the queue has or where the base rate sits — it asks only whether the model orders that client's
pages correctly. **It is positive for all 21 clients where it can be computed** — range 0.3438 to 0.7852, median
**0.4814**, and **zero** clients negative.

**So the segmented model is not justified, and for a better reason than ML-08 gave.** ML-08 showed
that data-rich clients gain nothing from the extra columns they have. This shows something stronger:
**clients do not differ in the relationship at all.** One model is correct because there is one
relationship, not because segmentation was impractical.

**What this costs, and it is worth stating in the recommendation.** Twelve of 27 clients receive a
queue that is simply their whole eligible page list. For them the product is the *gate*, not the
model — which matches ML-07's finding that the gate carries information and the ordering did not.
Selling those clients a "ranked queue" would misdescribe what they get.

> ⚠️ **Renamed by FlyRank's answer (2026-08-19).** The arithmetic here is untouched — twelve clients
> had fewer pages than slots, and `client_d211cb07b9059bab` had **72** against 100 — but *capacity
> artefact* is now the wrong name for it. With K a configurable per-client budget, a client receiving
> its whole page list is a **misconfigured budget**, not a capacity limit, and it is a settings bug
> rather than a structural fact about the product.
>
> That sharpens the closing paragraph rather than softening it. Those clients are not owed a smaller
> promise; they are owed a K set from their own pool size. Choosing K per client is ML-10's job.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit on the final five features

The same hunt as ML-05, on what actually ships: `peak_ratio`, `prior_trend`, `content_age_days`,
`impr_90d`, `avg_position`.

**The order matters.** A positive control runs *first*. If deliberately injecting the answer does not
send the score toward 1.0, the harness cannot detect leakage and every result below it is
uninterpretable — including the clean ones. A leakage audit that has never seen leakage has proved
nothing.

Then the attack checklist: timeline, ablation on the dominant feature, population selection, and
product flags.

In [6]:
# ---------------------------------------------------------------- 1. positive control
# Inject the answer. If the harness works, these must approach 1.0.
LEAKS = {
    "target itself": "target",
    "future_daily (the label's numerator)": "future_daily",
    "declined (the evaluation label)": "declined",
}
ctrl = []
for seed in range(5):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te]
    for label, col in [("none (the shipped model)", None)] + list(LEAKS.items()):
        feats = FULL if col is None else FULL + [col]
        sc_ = StandardScaler().fit(train[feats])
        m = Ridge(alpha=1.0).fit(sc_.transform(train[feats]), train["target"])
        pred = m.predict(sc_.transform(test[feats]))
        ctrl.append({"injected": label,
                     "spearman": rank_agreement(pred, test["target"].values, True),
                     "auc": roc_auc_score(test["declined"], -pred)})

cc = pd.DataFrame(ctrl).groupby("injected").mean().round(4)
print("POSITIVE CONTROL -- inject the answer, five grouped splits")
print(cc.to_string())
clean = cc.loc["none (the shipped model)", "spearman"]
full_leak = cc.loc["target itself", "spearman"]
label_leak_auc = cc.loc["declined (the evaluation label)", "auc"]
print()
print(f"harness detects a full leak: {full_leak > 0.99} (target injected -> {full_leak:.4f})")
print(f"harness detects a label leak: {label_leak_auc > 0.99} "
      f"(declined injected -> AUC {label_leak_auc:.4f})")
print(f"shipped model sits at {clean:.4f}, far below either")
print()
print("A partial leak leaks partially. future_daily reaches only "
      f"{cc.loc['future_daily (the label\'s numerator)', 'spearman']:.4f} because the target is")
print("asinh(future_daily) - asinh(baseline_daily): one term of two is not the answer.")
print("An earlier version of this cell demanded EVERY injection exceed 0.9 and")
print("printed 'harness works: False'. That criterion was wrong, not the harness.")
print()

# ---------------------------------------------------------------- 2. timeline
print("TIMELINE -- every feature must be knowable at the decision point")
DEC = pd.Timestamp(D1)
created = pd.to_datetime(df["content_created_date"], errors="coerce")
print(f"  content_created_date after {D1}: "
      f"{int((created > DEC).sum()):,} of {int(created.notna().sum()):,}")
print(f"  latest created_date: {created.max().date()}")
print("  impr_90d, avg_position, peak_ratio, prior_trend: all summed over")
print(f"    [{D1} - 90d, {D1}) by the query's WHERE clause -- no future rows reachable")
print(f"  target window: [{D1}, {D1} + 30d) -- strictly after, no overlap")
print()

# ---------------------------------------------------------------- 3. ablation
print("ABLATION -- drop each feature in turn, ten grouped splits")
abl = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te]
    for drop in [None] + FULL:
        feats = FULL if drop is None else [f for f in FULL if f != drop]
        sc_ = StandardScaler().fit(train[feats])
        m = Ridge(alpha=1.0).fit(sc_.transform(train[feats]), train["target"])
        pred = m.predict(sc_.transform(test[feats]))
        abl.append({"dropped": drop or "nothing",
                    "spearman": rank_agreement(pred, test["target"].values, True)})

ab = pd.DataFrame(abl).groupby("dropped")["spearman"].mean().round(4).sort_values()
print(ab.to_string())
print()

# ---------------------------------------------------------------- 4. population + flags
print("POPULATION SELECTION -- does the gate read the outcome window?")
gate_inputs = {"baseline_daily": "days 30-90 before the decision point",
               "content_age_days": "created_date, immutable, vs the decision date",
               "impr_90d": "the 90 days before the decision point",
               "slip": "recent 30d vs days 30-90, both before"}
for k, v in gate_inputs.items():
    print(f"  {k:18s} {v}")
print("  none reads the label window. The cohort keeps pages that reach zero,")
print("  so survival in the outcome month is NOT a selection criterion.")
print()
print("PRODUCT FLAGS -- FlyRank's own scores must never be inputs")
flagish = [c for c in ev.columns
           if any(t in c.lower() for t in ("flag", "risk", "health", "score", "priority",
                                           "optimiz", "eligible"))]
print(f"  flag-like columns present in the frame: {flagish}")
print(f"  any of them in FULL: {[f for f in FULL if f in flagish]}")

# ---------------------------------------------------------------- 5. the four-feature model
# The ablation says dropping content_age_days IMPROVES the model, which the
# permutation importance (-0.0645) predicted. Check it on the full metric
# suite before recommending a change to what ships.
LEAN = [f for f in FULL if f != "content_age_days"]
lean_rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te]
    for label, feats in [("FULL (5)", FULL), ("LEAN (4, no age)", LEAN)]:
        sc_ = StandardScaler().fit(train[feats])
        m = Ridge(alpha=1.0).fit(sc_.transform(train[feats]), train["target"])
        pred = m.predict(sc_.transform(test[feats]))
        hits = picks = 0
        for _, g in test.assign(s=pred).groupby("client_hash_id"):
            top = g.nsmallest(min(K, len(g)), "s")
            hits += int(top["declined"].sum()); picks += len(top)
        lean_rows.append({"features": label,
                          "spearman": rank_agreement(pred, test["target"].values, True),
                          "p_at_100": hits / picks,
                          "auc": roc_auc_score(test["declined"], -pred)})

ln = pd.DataFrame(lean_rows).groupby("features").mean().round(4)
print()
print("DROPPING content_age_days -- full metric suite, ten grouped splits")
print(ln.to_string())
print()
for m_ in ["spearman", "p_at_100", "auc"]:
    print(f"  {m_:10s} {ln.loc['LEAN (4, no age)', m_] - ln.loc['FULL (5)', m_]:+.4f}")

POSITIVE CONTROL -- inject the answer, five grouped splits
                                      spearman     auc
injected                                              
declined (the evaluation label)         0.8039  0.9993
future_daily (the label's numerator)    0.6596  0.8267
none (the shipped model)                0.5498  0.7625
target itself                           0.9999  1.0000

harness detects a full leak: True (target injected -> 0.9999)
harness detects a label leak: True (declined injected -> AUC 0.9993)
shipped model sits at 0.5498, far below either

A partial leak leaks partially. future_daily reaches only 0.6596 because the target is
asinh(future_daily) - asinh(baseline_daily): one term of two is not the answer.
An earlier version of this cell demanded EVERY injection exceed 0.9 and
printed 'harness works: False'. That criterion was wrong, not the harness.

TIMELINE -- every feature must be knowable at the decision point
  content_created_date after 2026-03-31: 0 of 202,0

dropped
peak_ratio          0.1192
impr_90d            0.5214
nothing             0.5302
prior_trend         0.5305
avg_position        0.5306
content_age_days    0.5546

POPULATION SELECTION -- does the gate read the outcome window?
  baseline_daily     days 30-90 before the decision point
  content_age_days   created_date, immutable, vs the decision date
  impr_90d           the 90 days before the decision point
  slip               recent 30d vs days 30-90, both before
  none reads the label window. The cohort keeps pages that reach zero,
  so survival in the outcome month is NOT a selection criterion.

PRODUCT FLAGS -- FlyRank's own scores must never be inputs
  flag-like columns present in the frame: ['last_optimized_date', 'optimization_eligible_date']
  any of them in FULL: []



DROPPING content_age_days -- full metric suite, ten grouped splits
                  spearman  p_at_100     auc
features                                    
FULL (5)            0.5302    0.7563  0.7584
LEAN (4, no age)    0.5546    0.7612  0.7693

  spearman   +0.0244
  p_at_100   +0.0049
  auc        +0.0109


**Verdict: no leakage found, and the audit removes a feature. The model ships with four, not five.**

**The positive control passes, which is what makes everything below it readable.**

| injected | spearman | AUC |
|---|---|---|
| **target itself** | **0.9999** | **1.0000** |
| `declined` (the evaluation label) | 0.8039 | **0.9993** |
| `future_daily` (one of the target's two terms) | 0.6596 | 0.8267 |
| **none — the shipped model** | **0.5498** | 0.7625 |

Injecting the answer produces near-perfection, so the harness can see leakage. The shipped model sits
at **0.5498** against a measured leak level of **0.9999** — not a near miss.

**A partial leak leaks partially, and that is the most useful thing this control taught.**
`future_daily` reaches only 0.6596, because `target = asinh(future_daily) − asinh(baseline_daily)` and
one term of two is not the answer. **A feature overlapping half the target's construction would not
announce itself loudly** — which is exactly the failure mode that cost this project three findings
before the null simulation caught it. Injection tests find blatant leaks; they do not find subtle ones.

> ⚠️ **This cell first printed `harness works: False`.** The criterion demanded *every* injection exceed
> 0.9, and `future_daily` does not. The criterion was wrong, not the harness. Recorded because a
> failing self-test that is itself broken is the worst possible thing to skim past.

**`peak_ratio` dominance is not leakage.** Dropping it collapses spearman from 0.5302 to **0.1192**,
which satisfies half the skill's rule — one feature towering over the rest. It fails the other half:
the rule requires a *near-perfect* score, and 0.5302 against a leak level of 0.9999 is nowhere near.
The null simulation already established the same thing from the other direction, with ~81% of the gain
surviving a future carrying no information.

**Timeline, population and flags are clean.** No `content_created_date` falls after the decision point
— **0 of 202,073**, latest value exactly 2026-03-31. Every windowed feature is bounded by the query's
`WHERE` clause. The gate reads `baseline_daily`, `content_age_days`, `impr_90d` and `slip`, all
pre-decision, and the cohort keeps pages that reach zero — so survival in the outcome month is not a
selection criterion. The two flag-like columns in the frame, `last_optimized_date` and
`optimization_eligible_date`, are excluded from the feature set.

**The audit changes what ships.**

| dropped | spearman |
|---|---|
| `peak_ratio` | 0.1192 |
| `impr_90d` | 0.5214 |
| nothing | 0.5302 |
| `prior_trend` | 0.5305 |
| `avg_position` | 0.5306 |
| **`content_age_days`** | **0.5546** |

Removing `content_age_days` improves every metric:

| | FULL (5) | LEAN (4) | change |
|---|---|---|---|
| spearman | 0.5302 | **0.5546** | **+0.0244** |
| `P@100` | 0.7563 | **0.7612** | +0.0049 |
| AUC | 0.7584 | **0.7693** | +0.0109 |

Direct ablation confirming what permutation importance predicted at **−0.0645**. Two independent
methods agree the feature is not merely useless but harmful — it correlates −0.1981 with the target
and is collinear enough with `peak_ratio` to split its coefficient.

**ML-08 audited twenty candidates down to five. Validation removes a sixth.** That is the process
working rather than failing: ML-08's comparison table never dropped features one at a time, because
it was comparing model classes rather than feature sets. The ablation is cheap and should have run
there.

### The one segment ML-08 never tested: GA4

ML-08 ran the segment protocol on three groups — keyword, links, structural. **GA4 was excluded by
argument rather than by test**, on the grounds that ML-06 Test 9 had already measured per-client
engagement correlations at a median of **+0.037** and **+0.015** and found 0 of 13 and 2 of 24 clients
above \|ρ\| = 0.2.

**That argument does not hold, for the reason this project has demonstrated more than any other:
correlation barely predicts contribution.** `prior_trend` correlates **+0.5438** with the target and
contributes **0.0038**; `url_char_count` correlates **−0.2225** and contributes **−0.0185**. Inferring
"will not help a model" from "does not correlate" is the inference ML-08 disproved five times over.

Test 9 also measured on ML-06's cohort, not the gated evaluation pool, and never fit a model. Three
differences from the protocol the other groups received.

**Same test as the others, on the four-feature model that now ships.**

In [7]:
GA4 = ["engaged_rate", "pages_per_session", "scroll_per_session", "log_ga4_sessions"]
GA4 = [c for c in GA4 if c in ev.columns]
assert GA4, "GA4 features not built -- an empty result here is a bug, not a finding"

have = ev[GA4].notna().all(axis=1) & (ev["ga4_sessions_90d"].fillna(0) > 0)
per = ev.assign(h=have).groupby("client_hash_id")["h"].mean()
rich = per[per > 0.95].index
sub = ev[ev["client_hash_id"].isin(rich) & have].copy()

print(f"GA4 features: {GA4}")
print(f"clients with >95% GA4 coverage: {len(rich)} of {ev['client_hash_id'].nunique()}")
print(f"pages in the GA4-rich segment: {len(sub):,} | decline rate {sub['declined'].mean():.4f}")
print()
for f in GA4:
    ok = sub[f].notna() & np.isfinite(sub[f])
    print(f"  Spearman({f:20s}, target) = "
          f"{sub.loc[ok, f].corr(sub.loc[ok, 'target'], method='spearman'):+.4f}")

if sub["client_hash_id"].nunique() < 8:
    print(f"\nonly {sub['client_hash_id'].nunique()} clients -- too few to hold out")
else:
    rows_g = []
    for seed in range(10):
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                      .split(sub, groups=sub["client_hash_id"]))
        train, test = sub.iloc[tr], sub.iloc[te]
        for label, feats in [("LEAN (shared model)", LEAN), ("LEAN + GA4", LEAN + GA4)]:
            t2, e2 = train.dropna(subset=feats), test.dropna(subset=feats)
            if e2.empty:
                continue
            sc_ = StandardScaler().fit(t2[feats])
            m = Ridge(alpha=1.0).fit(sc_.transform(t2[feats]), t2["target"])
            pred = m.predict(sc_.transform(e2[feats]))
            hits = picks = 0
            for _, g in e2.assign(s=pred).groupby("client_hash_id"):
                top = g.nsmallest(min(K, len(g)), "s")
                hits += int(top["declined"].sum()); picks += len(top)
            rows_g.append({"features": label,
                           "spearman": rank_agreement(pred, e2["target"].values, True),
                           "p_at_100": hits / picks,
                           "auc": roc_auc_score(e2["declined"], -pred)})

    gg = pd.DataFrame(rows_g).groupby("features").mean().round(4)
    print()
    print("inside the GA4-rich clients only, ten grouped splits:")
    print(gg.to_string())
    print()
    print("what GA4 buys, inside the clients that have it:")
    for m_ in ["spearman", "p_at_100", "auc"]:
        print(f"  {m_:10s} {gg.loc['LEAN + GA4', m_] - gg.loc['LEAN (shared model)', m_]:+.4f}")

# The ">95% coverage" figure above is measured on the GATED POOL, where the
# gate has already selected higher-traffic pages that are likelier to carry
# GA4. Reporting it alone invites the reading "only six clients have GA4",
# which is not what it says. The full-cohort distribution is the honest
# companion number.
full = df[df["impr_90d"] > 0].copy()
full["has_sessions"] = full["ga4_sessions_90d"].fillna(0) > 0
full["flag_on"] = full["ga4_available"].fillna(False)
big = (full.groupby("client_hash_id")
       .agg(pages=("content_hash_id", "size"),
            pct_sessions=("has_sessions", lambda s: s.mean() * 100),
            pct_flag=("flag_on", lambda s: s.mean() * 100)))
big = big[big["pages"] >= 200]

print()
print(f"FULL COHORT -- {len(big)} clients with >= 200 pages")
for lo, hi, name in [(99, 101, ">=99%"), (50, 99, "50-99%"),
                     (5, 50, "5-50%"), (-1, 5, "<5%")]:
    s = ((big["pct_sessions"] > lo) & (big["pct_sessions"] <= hi)).sum()
    f = ((big["pct_flag"] > lo) & (big["pct_flag"] <= hi)).sum()
    print(f"  pages with GA4 sessions {name:7s}: {s:2d} clients "
          f"(ga4_data_available flag agrees: {f:2d})")
print(f"  median per-client coverage: {big['pct_sessions'].median():.1f}%")
print()
print("No client has near-complete GA4. The median client carries it on under")
print("half its pages. The six-client figure is a strict threshold on a filtered")
print("population, not a count of clients that have GA4 at all.")

GA4 features: ['engaged_rate', 'pages_per_session', 'scroll_per_session', 'log_ga4_sessions']
clients with >95% GA4 coverage: 6 of 30
pages in the GA4-rich segment: 8,756 | decline rate 0.4076

  Spearman(engaged_rate        , target) = -0.0062
  Spearman(pages_per_session   , target) = -0.1841
  Spearman(scroll_per_session  , target) = -0.0886
  Spearman(log_ga4_sessions    , target) = -0.0635

only 6 clients -- too few to hold out



FULL COHORT -- 36 clients with >= 200 pages
  pages with GA4 sessions >=99%  :  0 clients (ga4_data_available flag agrees:  0)
  pages with GA4 sessions 50-99% : 17 clients (ga4_data_available flag agrees: 17)
  pages with GA4 sessions 5-50%  : 11 clients (ga4_data_available flag agrees: 11)
  pages with GA4 sessions <5%    :  8 clients (ga4_data_available flag agrees:  8)
  median per-client coverage: 46.2%

No client has near-complete GA4. The median client carries it on under
half its pages. The six-client figure is a strict threshold on a filtered
population, not a count of clients that have GA4 at all.


**Verdict: the GA4 segment cannot be tested at this scale — and the reason corrects ML-07.**

**GA4 is partial almost everywhere. Very few clients have it on most of their pages, and none have it
on nearly all.** Across the full cohort, 36 clients with at least 200 pages:

| pages carrying GA4 sessions | clients |
|---|---|
| 99% or more | **0** |
| 50–99% | **17** |
| 5–50% | 11 |
| under 5% | 8 |

Median per-client coverage is **46.2%**. The `ga4_data_available` flag agrees with `sessions > 0`
exactly in every band, so the flag was always the honest column to read.

**The six-client figure quoted below means something narrower than it sounds.** It counts clients with
**more than 95% coverage inside the gated evaluation pool** — a strict threshold on a population the
gate has already filtered toward higher-traffic pages, which carry GA4 more often. It is the number
that decides whether a segment is large enough to hold out. **It is not a count of clients that have
GA4 at all**, and reading it that way would suggest 24 clients have none, when 17 carry it on more
than half their pages.

**ML-07's check overstated coverage for a different reason**, and that one is worth keeping.
It reported GA4 present for 29 of 36 clients with 6 at zero, and this notebook repeated that as "24
clients could get a richer model". Both readings were wrong.

**That check measured `notna()`.** `SUM(ga4_sessions)` returns **0** for a page with no GA4 traffic,
not null — so "available" meant "the column exists", which is true of nearly every row. Requiring
actual sessions, which any engagement *rate* needs as a denominator, drops median per-client coverage
from effectively 100% to **46.2%**, and leaves only **6** clients above the 95% mark inside the pool.

**The reusable form of this:** where a rate is computed, "column present" and "data present" are
different questions, and a denominator of zero is not missing data. An availability audit that counts
nulls will not see it.

**So the segment fails the same bar backlinks failed at 4 clients.** Holding out 20% of six clients
leaves one, and training on five. Any number from that would be noise wearing a result's clothes.

**What the correlations show inside the 6 clients that do have it**, on 8,756 pages:

| feature | Spearman vs target |
|---|---|
| `pages_per_session` | **−0.1841** |
| `scroll_per_session` | −0.0886 |
| `log_ga4_sessions` | −0.0635 |
| `engaged_rate` | −0.0062 |

`pages_per_session` at −0.1841 is comparable to `content_age_days` cohort-wide, so this is not
obviously nothing — and it is exactly the kind of correlation ML-08 showed can contribute zero. It
cannot be resolved either way here.

**The honest position.** GA4 is excluded because **too few clients carry usable data to validate it**,
not because it was shown not to help. ML-06 Test 9 measured correlation on a different cohort and
never fit a model; this notebook's earlier claim that Test 9 settled the question was wrong. The
question is open and will stay open until more clients have GA4 instrumented.

**What FlyRank could act on.** The blocker is not that engagement fails to predict — that was never
established. It is that **no client has GA4 instrumented across its portfolio**, so there is no
segment large enough to validate the question on. Seventeen clients sit between 50% and 99%; raising
any of them toward full coverage would make this testable.

## 4. Claim rewrite

**The boldest sentence in this project**, from the capstone's interpretation section:

> **The label is dominated by the arithmetic of its own denominator.**

It is bold in the right way — it withdraws a finding rather than asserting one — but it still
overreaches in three places, and the safe-language rules in `writing-honest-claims` catch all three.

**"The label"** — which label, measured how, on what population? The sentence reads as a property of
the data. It is a property of *one* label definition on *one* cohort at *two* decision points.

**"Dominated"** — a strong word with no number attached in the sentence itself. The evidence is a null
producing **79.0** points of gradient against **12.6** observed, and Test 3's spread collapsing from
**46.2** to **5.3** once the `~was_declining` exclusion is removed. Those numbers belong in the claim,
not two paragraphs below it.

**"Its own denominator"** — asserts the mechanism. Two mechanisms were actually at work, and the
project first named the wrong one: Test 3's gradient turned out to be **89%** the label's exclusion
rule, with the shared denominator secondary.

**Rewritten:**

> Measured at two decision points on this cohort, the binary decline label was largely an artefact of
> how it was constructed rather than a description of page behaviour. Two mechanisms contributed. The
> label excluded already-declining pages by definition, which accounts for **89%** of the gradient
> Test 3 reported (spread **46.2** → **5.3** points when the exclusion is removed). Separately, the
> label divided by the same window used to select the cohort: a null carrying no information about
> the future produced **79.0** points of gradient against **12.6** observed on the same pages. Both
> findings reproduce at D1 and D2. They are properties of this label definition on this warehouse,
> and do not establish anything about decline in search generally.

Longer, and every clause is now checkable. **Observed** replaces asserted, the population and dates are
named, the mechanisms are separated and quantified, and the last sentence states what the claim does
*not* cover — which is the part a reader would otherwise supply themselves, generously.

In [8]:
# Section 4 is a writing exercise -- the numbers it cites are all
# produced by cells in ML-06 and quoted here, not recomputed.
print("no computation in this section; see ML-06 Test 3 and the null simulation")

no computation in this section; see ML-06 Test 3 and the null simulation


## 5. The two things the capstone said ML-09 still owed

Section 5 of the capstone ends: *"What ML-09 still owes: three concrete failure cases read
individually, whether the 0.458 client is data-poor or structurally different, and whether a ranking
objective closes boosting's gap."*

The middle one is settled in §2a — the 0.458 client has 72 pages against 100 slots, so the queue took
every page and precision equalled its base rate. The other two are here.

**The failure cases.** ML-08 printed three confident picks that grew instead. Printing them is not
reading them, and the skill asks for the action, the reason, and what would make it wrong.

**The ranking objective.** ML-06 proposed two follow-ups for boosting: `LGBMRanker` with LambdaRank,
and boosting on a rank-transformed target. `lightgbm` is not installed and adding a dependency for one
test is a poor trade, so the second is run here — it tests the same hypothesis, that boosting
underperforms because squared error on a heavy-tailed target is the wrong objective for a ranking
problem.

In [9]:
# ---------------------------------------------------------------- the three failures
# Re-derive them here rather than quoting ML-08, so the reading is reproducible.
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
              .split(ev, groups=ev["client_hash_id"]))
train, test = ev.iloc[tr], ev.iloc[te].copy()
sc_ = StandardScaler().fit(train[LEAN])
m = Ridge(alpha=1.0).fit(sc_.transform(train[LEAN]), train["target"])
test["pred"] = m.predict(sc_.transform(test[LEAN]))

confident = test.nsmallest(400, "pred")
wrong = confident[confident["target"] > 0].nsmallest(3, "pred")
cols = ["pred", "target", "peak_ratio", "prior_trend", "impr_90d",
        "baseline_daily", "recent_daily", "future_daily"]
print("three most confident picks that grew instead:")
print(wrong[cols].to_string(index=False, float_format=lambda v: f"{v:,.3f}"))
print()
print(f"all three from the same client: "
      f"{wrong['client_hash_id'].nunique() == 1}")
print(f"all three fell below baseline before the decision, then recovered above it: "
      f"{bool(((wrong['recent_daily'] < wrong['baseline_daily']) & (wrong['future_daily'] > wrong['baseline_daily'])).all())}")
print()

# Is that pattern general? Confident picks, split by how far the page had fallen.
confident = confident.assign(
    band=pd.cut(confident["peak_ratio"], [0, 0.5, 0.75, 1.0, 1.5, 10**9],
                labels=["<0.5", "0.5-0.75", "0.75-1.0", "1.0-1.5", ">1.5"]))
bands = (confident.groupby("band", observed=True)
         .agg(n=("declined", "size"), precision=("declined", "mean")).round(3))
print("how often a confident decline pick is right, by peak_ratio:")
print(bands.to_string())
print()

# ---------------------------------------------------------------- rank objective
from sklearn.ensemble import HistGradientBoostingRegressor

rank_rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te]
    y_raw = train["target"]
    y_rank = train["target"].rank(pct=True)
    for label, y in [("boosting on the raw target", y_raw),
                     ("boosting on rank(target)", y_rank)]:
        mm = HistGradientBoostingRegressor(random_state=SEED).fit(train[LEAN], y)
        pred = mm.predict(test[LEAN])
        hits = picks = 0
        for _, g in test.assign(s=pred).groupby("client_hash_id"):
            top = g.nsmallest(min(K, len(g)), "s")
            hits += int(top["declined"].sum()); picks += len(top)
        rank_rows.append({"objective": label,
                          "spearman": rank_agreement(pred, test["target"].values, True),
                          "p_at_100": hits / picks,
                          "auc": roc_auc_score(test["declined"], -pred)})
    # ridge on LEAN for the reference line
    s2 = StandardScaler().fit(train[LEAN])
    r = Ridge(alpha=1.0).fit(s2.transform(train[LEAN]), y_raw)
    pr = r.predict(s2.transform(test[LEAN]))
    hits = picks = 0
    for _, g in test.assign(s=pr).groupby("client_hash_id"):
        top = g.nsmallest(min(K, len(g)), "s")
        hits += int(top["declined"].sum()); picks += len(top)
    rank_rows.append({"objective": "ridge on the raw target (shipped)",
                      "spearman": rank_agreement(pr, test["target"].values, True),
                      "p_at_100": hits / picks,
                      "auc": roc_auc_score(test["declined"], -pr)})

rr = pd.DataFrame(rank_rows).groupby("objective").mean().round(4)
print("does a ranking objective close boosting's gap? ten grouped splits, LEAN features")
print(rr.to_string())
print()
d = (rr.loc["boosting on rank(target)", "spearman"]
     - rr.loc["boosting on the raw target", "spearman"])
print(f"rank transform buys boosting {d:+.4f} spearman")
print(f"boosting still trails ridge by "
      f"{rr.loc['ridge on the raw target (shipped)', 'spearman'] - rr.loc['boosting on rank(target)', 'spearman']:+.4f}")
print()
print("what the rank transform buys boosting, every metric:")
for m_ in ["spearman", "p_at_100", "auc"]:
    print(f"  {m_:10s} "
          f"{rr.loc['boosting on rank(target)', m_] - rr.loc['boosting on the raw target', m_]:+.4f}")

three most confident picks that grew instead:
  pred  target  peak_ratio  prior_trend  impr_90d  baseline_daily  recent_daily  future_daily
-0.707   0.001       0.609       -0.633 2,519.000          33.467        17.033        33.500
-0.706   0.266       0.607       -0.610 5,077.000          67.500        34.233        88.067
-0.703   0.224       0.619       -0.505 2,404.000          31.800        16.533        39.800

all three from the same client: False
all three fell below baseline before the decision, then recovered above it: True

how often a confident decline pick is right, by peak_ratio:
            n  precision
band                    
0.5-0.75  400      0.932



does a ranking objective close boosting's gap? ten grouped splits, LEAN features
                                   spearman  p_at_100     auc
objective                                                    
boosting on rank(target)             0.5378    0.7533  0.7652
boosting on the raw target           0.5374    0.7344  0.7605
ridge on the raw target (shipped)    0.5546    0.7612  0.7693

rank transform buys boosting +0.0004 spearman
boosting still trails ridge by +0.0168

what the rank transform buys boosting, every metric:
  spearman   +0.0004
  p_at_100   +0.0189
  auc        +0.0047


**Both delivered. The failure cases have one shared cause, and the ranking-objective hypothesis is
not supported.**

### The three failures are all recoveries

| pred | target | peak_ratio | prior_trend | baseline/day | recent/day | future/day |
|---|---|---|---|---|---|---|
| −0.707 | 0.001 | 0.609 | −0.633 | 33.5 | 17.0 | 33.5 |
| −0.706 | 0.266 | 0.607 | −0.610 | 67.5 | 34.2 | **88.1** |
| −0.703 | 0.224 | 0.619 | −0.505 | 31.8 | 16.5 | 39.8 |

**All three fell to roughly half their baseline before the decision point, then recovered above it.**
The model saw a page in freefall and extrapolated the fall. Unlike ML-08's version of this table, they
are **not** all from one client — so it is a pattern, not a client quirk.

**The action for each is the same: no action, and that is the finding.** A specialist sent these three
would review pages that were already fixing themselves. **What would make it wrong:** if the recovery
were caused by an intervention inside the outcome window, these would be successes rather than
failures — and nothing in this data distinguishes those cases, since `last_optimized_date` is
100% post-decision and unusable.

**This is the recovery quadrant, and ML-08 already measured it as the weakest.** The lane test scored
recovery at **+0.0962** over random against decline's **+0.2111**. These three are that number in
individual form rather than a separate defect.

### The model does not make confident calls outside its reliable regime

| peak_ratio band | n | precision |
|---|---|---|
| 0.5–0.75 | **400** | **0.932** |

All 400 of the most confident picks fall in a single band, where the model is right **93.2%** of the
time. Since `peak_ratio` dominates the model, confidence and `peak_ratio` are nearly the same
quantity — so the model is structurally incapable of being confident about pages far from that band.
That is a better property than the guardrail I expected to need: there are no confident picks in the
regime where confident picks would be wrong.

### The ranking objective does not close boosting's gap

| objective | spearman | `P@100` | AUC |
|---|---|---|---|
| **ridge on the raw target (shipped)** | **0.5546** | **0.7612** | **0.7693** |
| boosting on rank(target) | 0.5378 | 0.7533 | 0.7652 |
| boosting on the raw target | 0.5374 | 0.7344 | 0.7605 |

**The rank transform buys boosting +0.0004 spearman.** ML-06 proposed it on the hypothesis that
boosting underperforms because squared error on a heavy-tailed target is the wrong objective for a
ranking problem. **That hypothesis is not supported** — fixing the objective changes rank agreement by
four ten-thousandths, and boosting still trails ridge by **0.0168**.

It does help the threshold metrics: `P@100` gains **+0.0189** and AUC **+0.0047**. So the transform is
doing something real at the top of the queue while leaving the overall ordering untouched — which is
the opposite pattern to the log transforms tested in ML-08, and not what an objective-mismatch story
predicts either.

**The surviving explanation is the other one ML-06 offered:** the relationship is close to linear,
`peak_ratio` carries it almost alone, and trees pay variance for flexibility they do not need. That
was stated there as the less interesting of two candidates. It is now the only one standing.

**`LGBMRanker` with LambdaRank remains untested** — `lightgbm` is not installed, and adding a project
dependency to test a hypothesis the cheaper experiment just failed to support is a poor trade. Left
open and named rather than quietly dropped.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 6. The hyperparameters nobody checked

Every model in this project ran on defaults. `Ridge(alpha=1.0)`, `LogisticRegression` with `C` left
at 1.0, `HistGradientBoostingRegressor` with nothing set but a seed. ML-08 noted this once — *"a
negative result here means 'not on defaults', not 'impossible'"* — and then never acted on it.

That caveat carries more weight than it was given. **Boosting's third-place finish and several
feature rejections tested on trees both rest on untuned models.** If the defaults were handicapping
the trees, conclusions drawn from them are conclusions about sklearn's defaults.

**Tuning has to happen inside the training fold**, or the held-out clients influence the choice and
the comparison stops being honest. Two forms of that here:

- **`RidgeCV`** picks `alpha` by efficient leave-one-out on the training data only. No extra splits,
  no contamination.
- **Boosting** gets a small grid, scored on an *inner* grouped split carved out of the training
  clients. The outer test clients are never seen.

The question is not whether tuned models score higher on their own training data — they will. It is
whether the **ordering** of the comparison changes on held-out clients.

In [10]:
def score(pred, test):
    """Every metric this project reports, for one prediction vector."""
    hits = picks = 0
    for _, g in test.assign(s=pred).groupby("client_hash_id"):
        top = g.nsmallest(min(K, len(g)), "s")
        hits += int(top["declined"].sum()); picks += len(top)
    return {"spearman": rank_agreement(pred, test["target"].values, True),
            "p_at_100": hits / picks,
            "auc": roc_auc_score(test["declined"], -pred)}


from sklearn.linear_model import RidgeCV

ALPHAS = np.logspace(-3, 4, 15)
GRID = [{"learning_rate": lr, "max_leaf_nodes": ln, "max_iter": it}
        for lr in (0.03, 0.1) for ln in (7, 31) for it in (100, 400)]

tuned_rows, picked_alpha, picked_cfg = [], [], []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te]

    sc_ = StandardScaler().fit(train[LEAN])
    Xtr, Xte = sc_.transform(train[LEAN]), sc_.transform(test[LEAN])

    # --- ridge, default vs leave-one-out tuned inside the training fold
    for label, model in [("ridge alpha=1.0 (shipped)", Ridge(alpha=1.0)),
                         ("ridge, alpha tuned in-fold", RidgeCV(alphas=ALPHAS))]:
        m = model.fit(Xtr, train["target"])
        if hasattr(m, "alpha_"):
            picked_alpha.append(m.alpha_)
        pred = m.predict(Xte)
        tuned_rows.append({"model": label, **score(pred, test)})

    # --- boosting, default vs grid picked on an INNER grouped split
    itr, ite = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
                    .split(train, groups=train["client_hash_id"]))
    inner_tr, inner_va = train.iloc[itr], train.iloc[ite]
    best, best_cfg = -9, None
    for cfg in GRID:
        mm = HistGradientBoostingRegressor(random_state=SEED, **cfg).fit(
            inner_tr[LEAN], inner_tr["target"])
        rho = rank_agreement(mm.predict(inner_va[LEAN]), inner_va["target"].values, True)
        if rho > best:
            best, best_cfg = rho, cfg
    picked_cfg.append(str(best_cfg))
    for label, mm in [("boosting, defaults", HistGradientBoostingRegressor(random_state=SEED)),
                      ("boosting, tuned in-fold",
                       HistGradientBoostingRegressor(random_state=SEED, **best_cfg))]:
        m = mm.fit(train[LEAN], train["target"])
        tuned_rows.append({"model": label, **score(m.predict(test[LEAN]), test)})

tt = pd.DataFrame(tuned_rows).groupby("model").mean().round(4).sort_values(
    "spearman", ascending=False)
print(tt.to_string())
print()
print(f"alphas RidgeCV chose across seeds: "
      f"{sorted(set(round(a, 3) for a in picked_alpha))}")
print(f"boosting configs chosen: {len(set(picked_cfg))} distinct across 10 seeds")
for cfg in sorted(set(picked_cfg)):
    print(f"   {picked_cfg.count(cfg)}x  {cfg}")
print()
for a, b, name in [("ridge alpha=1.0 (shipped)", "ridge, alpha tuned in-fold", "ridge"),
                   ("boosting, defaults", "boosting, tuned in-fold", "boosting")]:
    for m_ in ["spearman", "p_at_100", "auc"]:
        print(f"  {name:9s} {m_:10s} tuning buys {tt.loc[b, m_] - tt.loc[a, m_]:+.4f}")

# Does tuning change the ORDERING, which is the only thing that matters here?
gap_default = tt.loc["ridge alpha=1.0 (shipped)", "spearman"] - tt.loc["boosting, defaults", "spearman"]
gap_tuned = tt.loc["ridge alpha=1.0 (shipped)", "spearman"] - tt.loc["boosting, tuned in-fold", "spearman"]
print()
print(f"ridge lead over boosting, defaults: {gap_default:+.4f}")
print(f"ridge lead over boosting, tuned:    {gap_tuned:+.4f}")
print(f"tuning closes the gap by {(1 - gap_tuned / gap_default):.0%}, and does not reverse it")
print()
import collections
lrs = collections.Counter(eval(c)["learning_rate"] for c in picked_cfg)
leaves = collections.Counter(eval(c)["max_leaf_nodes"] for c in picked_cfg)
print(f"learning rates the inner split preferred: {dict(lrs)}")
print(f"leaf counts the inner split preferred:    {dict(leaves)}")
print("The tuner asks for a simpler model every time -- lower learning rate,")
print("shallower trees. It is trying to make the boosting model more linear.")

                            spearman  p_at_100     auc
model                                                 
ridge, alpha tuned in-fold    0.5546    0.7612  0.7693
ridge alpha=1.0 (shipped)     0.5546    0.7612  0.7693
boosting, tuned in-fold       0.5494    0.7460  0.7679
boosting, defaults            0.5374    0.7344  0.7605

alphas RidgeCV chose across seeds: [0.001, 10.0, 31.623, 100.0]
boosting configs chosen: 4 distinct across 10 seeds
   2x  {'learning_rate': 0.03, 'max_leaf_nodes': 31, 'max_iter': 100}
   1x  {'learning_rate': 0.03, 'max_leaf_nodes': 31, 'max_iter': 400}
   5x  {'learning_rate': 0.03, 'max_leaf_nodes': 7, 'max_iter': 100}
   2x  {'learning_rate': 0.03, 'max_leaf_nodes': 7, 'max_iter': 400}

  ridge     spearman   tuning buys +0.0000
  ridge     p_at_100   tuning buys +0.0000
  ridge     auc        tuning buys +0.0000
  boosting  spearman   tuning buys +0.0120
  boosting  p_at_100   tuning buys +0.0116
  boosting  auc        tuning buys +0.0074

ridge lead over

**Verdict: ridge is insensitive to tuning, boosting is not, and what boosting asks for is the
finding.**

| model | spearman | `P@100` | AUC |
|---|---|---|---|
| ridge, alpha tuned in-fold | **0.5546** | **0.7612** | **0.7693** |
| **ridge, alpha = 1.0 (shipped)** | **0.5546** | **0.7612** | **0.7693** |
| boosting, tuned in-fold | 0.5494 | 0.7460 | 0.7679 |
| boosting, defaults | 0.5374 | 0.7344 | 0.7605 |

**Ridge: tuning buys +0.0000 on every metric.** `RidgeCV` chose alphas of **0.001, 10.0, 31.623 and
100.0** across the ten seeds — five orders of magnitude of disagreement about the right penalty,
producing identical numbers to four decimal places.

That is what a model looks like when regularisation cannot bite: 45,095 rows against four
standardised features, so the data overwhelms any reasonable prior. `alpha = 1.0` was not a lucky
guess; the parameter has no room to matter. **The shipped model carries no tuning risk.**

**Boosting: tuning buys +0.0120 spearman**, narrowing ridge's lead from **+0.0172** to **+0.0052**.
Tuning closes **70%** of the gap and does not reverse it. ML-08's model choice survives, and the
"not on defaults" caveat it carried is now discharged rather than outstanding.

**What the tuner asked for is the more useful result.**

| parameter | what the inner split chose |
|---|---|
| `learning_rate` | **0.03 in 10 of 10 seeds** — always the lower option |
| `max_leaf_nodes` | **7 in 7 of 10** — usually the shallower option |

Every seed wanted a slower learner. Most wanted shallower trees. **The tuner is consistently trying
to make the boosting model simpler — which is to say, more linear.**

That is independent support for the explanation ML-06 offered and called the less interesting of two
candidates: the relationship is close to linear, `peak_ratio` carries it almost alone, and trees pay
variance for flexibility they do not need. It has now survived three tests that could have killed it —
the null simulation, the failed ranking-objective experiment, and a hyperparameter search reaching for
it from a completely different direction.

**What this does not license.** The grid is small (two values per parameter, eight combinations) and
the inner split is a single 25% carve-out of the training clients rather than a full inner CV. A
larger search might find more. What it does establish is that boosting's third place was **not** an
artefact of leaving the defaults alone, which is the specific doubt ML-08 recorded and could not
resolve.

## 7. Two things this audit found late: the flag has three states, and nobody swept the raw table

Both came from questions asked after ML-09 was written, and both are corrections to work already
committed.

**The GA4 flag was reported as binary and is not.** Section 6 above states that `ga4_data_available`
agrees with `ga4_sessions > 0` "exactly in every band". That was measured at client-aggregate level,
where a few thousand rows out of twenty-five million disappear into rounding. At page-day level the
two disagree, and there is a third state neither of them mentions.

**Every distribution in this project was computed on the cohort, never on the raw table.** ML-06 §1
describes each numeric field, but the cohort is already 90-day aggregates — one row per page, not per
page-day. A single day carrying an impossible value is invisible once summed. Nothing swept the fact
table itself for extremes, which is a gap in the data contract rather than in the model.

In [11]:
import duckdb as _duck

RAW = REL  # the same fact-table relation the cohort is built from

print("=== ga4_data_available has three states, not two ===")
q_flag = f"""
SELECT CASE WHEN ga4_data_available IS NULL THEN 'NULL'
            WHEN ga4_data_available THEN 'TRUE' ELSE 'FALSE' END AS flag,
       COUNT(*) AS page_days,
       SUM(CASE WHEN COALESCE(ga4_sessions, 0) > 0 THEN 1 ELSE 0 END) AS with_sessions
FROM {RAW} GROUP BY 1 ORDER BY 2 DESC"""
fl = con.sql(q_flag).df()
fl["pct_of_rows"] = (fl["page_days"] / fl["page_days"].sum() * 100).round(2)
fl["zero_session_rows"] = fl["page_days"] - fl["with_sessions"]  # only meaningful for TRUE
print(fl.to_string(index=False))
print()
tr = fl[fl["flag"] == "TRUE"].iloc[0]
print(f"flag TRUE with zero sessions: {int(tr['zero_session_rows']):,} page-days "
      f"({tr['zero_session_rows'] / tr['page_days']:.2%} of flag-true rows)")
print("For FALSE and NULL the same column is just the row count -- those pages\n"
      "are not tracked, so 'zero sessions' says nothing about them.")
nullpct = fl.loc[fl["flag"] == "NULL", "pct_of_rows"].iloc[0]
print(f"So the flag DOES separate 'tracked, nobody visited' from 'not tracked' --")
print(f"but only for 1.4% of flag-true rows, and {nullpct:.2f}% of all rows are")
print("NULL, a third state meaning neither.")
print()

print("=== extremes in the RAW page-day table, which no distribution here had checked ===")
num = [c for c, t in con.sql(f"DESCRIBE SELECT * FROM {RAW}").df()[
       ["column_name", "column_type"]].values if t in ("BIGINT", "DOUBLE", "INTEGER")]
rows = []
for c in num:
    r = con.sql(f"""SELECT '{c}' AS col, MEDIAN({c}) AS med,
                    QUANTILE_CONT({c}, 0.99) AS p99, MAX({c}) AS max
                    FROM {RAW} WHERE {c} IS NOT NULL""").df()
    rows.append(r)
ex = pd.concat(rows, ignore_index=True)
ex["max_over_p99"] = (ex["max"] / ex["p99"].replace(0, np.nan)).round(1)
print(ex.sort_values("max_over_p99", ascending=False).head(8).to_string(index=False))
print()
top = ex.sort_values("max_over_p99", ascending=False).head(3)
for _, w in top.iterrows():
    used = w["col"] in FULL or w["col"] in LEAN
    note = ""
    if w["col"].endswith("_sec"):
        note = f"  =  {w['max'] / 3600:.1f} hours on one page in one day"
    print(f"{w['col']:26s} max {w['max']:>9,.0f}  p99 {w['p99']:>5,.0f}  "
          f"{w['max_over_p99']:>9,.0f}x   used by this project: {used}{note}")
print()
print("An earlier version of this cell divided whichever column topped the list")
print("by 3600 and called the result hours. The top column is a session COUNT,")
print("so it printed '4.6 hours' for a number that is not a duration. Only the")
print("_sec column is converted now.")

=== ga4_data_available has three states, not two ===


 flag  page_days  with_sessions  pct_of_rows  zero_session_rows
FALSE   38858117            0.0        58.30         38858117.0
 NULL   25095265            0.0        37.65         25095265.0
 TRUE    2693029      2655210.0         4.04            37819.0

flag TRUE with zero sessions: 37,819 page-days (1.40% of flag-true rows)
For FALSE and NULL the same column is just the row count -- those pages
are not tracked, so 'zero sessions' says nothing about them.
So the flag DOES separate 'tracked, nobody visited' from 'not tracked' --
but only for 1.4% of flag-true rows, and 37.65% of all rows are
NULL, a third state meaning neither.

=== extremes in the RAW page-day table, which no distribution here had checked ===


                     col  med  p99     max  max_over_p99
         sessions_direct  0.0  1.0 16537.0       16537.0
ga4_total_engagement_sec  0.0  2.0 32076.0       16038.0
           ga4_pageviews  0.0  5.0 64750.0       12950.0
               ga4_users  0.0  3.0 33755.0       11251.7
            ga4_sessions  0.0  3.0 19425.0        6475.0
              gsc_clicks  0.0  2.0  9558.0        4779.0
        sessions_organic  0.0  3.0  9416.0        3138.7
           scroll_events  0.0  1.0  3053.0        3053.0

sessions_direct            max    16,537  p99     1     16,537x   used by this project: False
ga4_total_engagement_sec   max    32,076  p99     2     16,038x   used by this project: False  =  8.9 hours on one page in one day
ga4_pageviews              max    64,750  p99     5     12,950x   used by this project: False

An earlier version of this cell divided whichever column topped the list
by 3600 and called the result hours. The top column is a session COUNT,
so it printed '4.6 ho

**Verdict: both corrections stand, the flag is three-valued, and four columns carry impossible
page-day values. Nothing here changed a result, and one of them easily could have.**

**`ga4_data_available` has three states, and tracking is rarer than any earlier section implied.**

| flag | page-days | share | with sessions |
|---|---|---|---|
| FALSE | 38,858,117 | **58.30%** | 0 |
| NULL | 25,095,265 | **37.65%** | 0 |
| **TRUE** | 2,693,029 | **4.04%** | 2,655,210 |

**Only about one page-day in twenty-five has GA4 tracking on.** The flag is TRUE with zero sessions
for **37,819** page-days — **1.40%** of flag-true rows — so it does separate "tracked, nobody visited"
from "not tracked", but thinly. The **37.65%** NULL group is a third state meaning neither, and no row
in it or in FALSE ever carries a session.

**This is what ">95% coverage" was counting.** Pages with at least one recorded session. The rest
recorded **zero**, and for the NULL group the data cannot say whether that is absent tracking or
absent visitors. Section 6's claim that the flag and `sessions > 0` agree "exactly" holds only at
client-aggregate level, where 37,819 rows out of sixty-six million vanish into rounding.

**Four raw columns exceed 10,000x their own 99th percentile.**

| column | p99 | max | ratio | used here |
|---|---|---|---|---|
| `sessions_direct` | 1 | **16,537** | **16,537x** | no |
| `ga4_total_engagement_sec` | 2 | 32,076 | 16,038x | no |
| `ga4_pageviews` | 5 | **64,750** | 12,950x | no |
| `ga4_users` | 3 | 33,755 | 11,251x | no |

**32,076 seconds is 8.9 hours on one page in one day.** 64,750 pageviews on a single page-day against
a p99 of five is the same kind of number. These are tabs left open, bots, or instrumentation faults —
not readers.

**None reached a feature, and that is luck rather than design.** All four are GA4 columns, and this
project built `engaged_rate`, `pages_per_session` and `scroll_per_session` from GA4. `pages_per_session`
divides `ga4_pageviews` by `ga4_sessions` — the numerator of that ratio is one of the four. It was
tested and dropped for lack of coverage, never for containing a 64,750-pageview day.

**The gap is structural, not incidental.** ML-06 §1 describes every numeric field of the *cohort*,
which is already 90-day sums. A single impossible day disappears once summed. **No distribution
anywhere in this project looked at the fact table itself.** That belongs in the data contract, where
it costs one query, rather than surfacing in week eight because someone asked whether any column had
an excessive value.

**Two bugs in this cell, both kept in the record.** It first computed "tracked but idle" for every
flag state, printing 38.8m FALSE rows under a label that only means anything for TRUE. And the
seconds-to-hours line converted whichever column topped the ranking — on three months of data that
was the `_sec` column, on seven months it is `sessions_direct`, a session **count**, so it printed
"4.6 hours" for a number that is not a duration. Code written around an expected answer mislabels a
different one when the data moves.

## 8. Walk-forward: is the one time-based number in this project trustworthy?

Every split used so far groups by **client** — it asks whether the model works on a client it has
never seen. Exactly one test asks whether it works on a *month* it has never seen: ML-08 §5 trains at
D1 and scores once at D2, retaining **53%** of its advantage over random.

The leakage skill is blunt about this: a time split is *"the only split that mimics deployment for
anything trend-like"*, and predicting next month's traffic is trend-like. **One step is the weakest
possible version of that test.** A single number cannot distinguish a stable 53% from an accident of
the particular gap between March and May.

**What the warehouse allows.** Daily rows run 2025-12-01 to 2026-06-30. Each decision point needs 90
days of history and a 30-day outcome window, so valid decision points span **2026-03-01 to
2026-06-01** — 92 days of room. Spaced monthly that is four points and three forward steps.

**A reproducibility bug found while writing this up.** The first version of `cohort_at` returned its
rows in whatever order DuckDB'''s parallel aggregation produced. `queue_precision` sorts by score and
breaks ties by row position, and the random baseline assigns its draws positionally, so the same code
on the same data returned a slightly different lift on every run -- 0.2449 one run, 0.2424 the next.
Small enough to look like rounding and large enough to make every figure below unciteable. ML-08'''s
equivalent query ends `ORDER BY p.content_hash_id`; this one had dropped it. Restored, and the numbers
below are now stable across runs.

**Stated before running it: the folds are not independent.** Consecutive 90-day windows share 60
days, and consecutive cohorts are largely the same pages. Three overlapping steps cannot establish a
trend. What they can do is show whether 53% is typical or exceptional, which is the only question
worth asking of a single number.

In [12]:
# queue_metrics lives in ML-08; ML-09 stands alone, so define it here rather
# than depend on a name that happens to exist in another notebook.
def queue_precision(frame, score_col, k, ascending):
    hits = picks = 0
    for _, g in frame.groupby("client_hash_id"):
        top = g.sort_values(score_col, ascending=ascending).head(min(k, len(g)))
        hits += int(top["declined"].sum()); picks += len(top)
    return hits / picks if picks else np.nan



def cohort_at(dstr):
    """The ML-08 cohort and frozen gate, rebuilt at an arbitrary decision date."""
    q = f"""
    WITH prior AS (
      SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(gsc_impressions) AS impr_90d,
        SUM(gsc_sum_position) AS sum_position_90d,
        SUM(gsc_impressions) FILTER (
            WHERE report_date < DATE '{dstr}' - INTERVAL 30 DAY) AS older60_impr,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{dstr}' - INTERVAL 30 DAY) AS recent30_impr,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{dstr}' - INTERVAL 60 DAY
              AND report_date <  DATE '{dstr}' - INTERVAL 30 DAY) AS base30_impr
      FROM {REL}
      WHERE report_date >= DATE '{dstr}' - INTERVAL 90 DAY AND report_date < DATE '{dstr}'
      GROUP BY content_hash_id HAVING SUM(gsc_impressions) > 0),
    fut AS (
      SELECT content_hash_id, SUM(gsc_impressions) AS future_impr FROM {REL}
      WHERE report_date >= DATE '{dstr}' AND report_date < DATE '{dstr}' + INTERVAL 30 DAY
      GROUP BY content_hash_id)
    SELECT p.*, COALESCE(f.future_impr, 0) AS future_impr
    FROM prior p LEFT JOIN fut f USING (content_hash_id)
    ORDER BY p.content_hash_id"""
    x = con.sql(q).df()
    for c in ["older60_impr", "recent30_impr", "base30_impr"]:
        x[c] = x[c].fillna(0)
    dd = pd.Timestamp(dstr)
    x = x.merge(con.sql(f"SELECT content_hash_id, content_created_date FROM "
                        f"read_parquet('{dim_file}')").df(), on="content_hash_id", how="left")
    x["baseline_daily"] = x["older60_impr"] / 60
    x["recent_daily"] = x["recent30_impr"] / 30
    x["future_daily"] = x["future_impr"] / 30
    x["target"] = np.arcsinh(x["future_daily"]) - np.arcsinh(x["baseline_daily"])
    x["declined"] = x["target"] < 0
    x["avg_position"] = x["sum_position_90d"] / x["impr_90d"].replace(0, np.nan) + 1
    x["content_age_days"] = (dd - pd.to_datetime(x["content_created_date"])).dt.days
    x["slip"] = np.where(x["baseline_daily"] > 0,
                         (x["baseline_daily"] - x["recent_daily"]) / x["baseline_daily"], np.nan)
    x["peak_ratio"] = np.where(x["impr_90d"] > 0,
                               x["recent_daily"] / (x["impr_90d"] / 90), np.nan)
    x["prior_trend"] = np.where(x["base30_impr"] > 0,
                                (x["recent30_impr"] - x["base30_impr"]) / x["base30_impr"], np.nan)
    med = x.groupby("client_hash_id")["impr_90d"].transform("median")
    gate = ((x["baseline_daily"] > 0) & (x["content_age_days"] >= 180)
            & (x["impr_90d"] >= med) & (x["slip"].fillna(0) <= 0.5))
    return x[gate].dropna(subset=LEAN).copy()


POINTS = ["2026-03-01", "2026-03-31", "2026-05-01", "2026-06-01"]
coh = {}
for d in POINTS:
    coh[d] = cohort_at(d)
    print(f"{d}: {len(coh[d]):>6,} pages | {coh[d]['client_hash_id'].nunique():>2} clients "
          f"| decline rate {coh[d]['declined'].mean():.4f}")

print()
walk = []
for train_d, test_d in zip(POINTS, POINTS[1:]):
    tr, te = coh[train_d], coh[test_d]
    sc_ = StandardScaler().fit(tr[LEAN])
    m = Ridge(alpha=1.0).fit(sc_.transform(tr[LEAN]), tr["target"])
    pred = m.predict(sc_.transform(te[LEAN]))
    rng_ = np.random.default_rng(0)
    p_model = queue_precision(te.assign(s=pred), "s", K, True)
    p_rand = np.mean([queue_precision(te.assign(s=rng_.random(len(te))), "s", K, False)
                      for _ in range(20)])
    # the same model scored on its own training month, for the in-sample reference
    p_self = queue_precision(tr.assign(s=m.predict(sc_.transform(tr[LEAN]))), "s", K, True)
    r_self = np.mean([queue_precision(tr.assign(s=rng_.random(len(tr))), "s", K, False)
                      for _ in range(20)])
    walk.append({"train": train_d, "test": test_d,
                 "test_base_rate": te["declined"].mean(),
                 "gap_days": (pd.Timestamp(test_d) - pd.Timestamp(train_d)).days,
                 "in_sample_lift": p_self - r_self,
                 "forward_lift": p_model - p_rand,
                 "retained": (p_model - p_rand) / (p_self - r_self)})

w = pd.DataFrame(walk)
print(w.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print(f"forward lift: mean {w['forward_lift'].mean():.4f} | "
      f"range {w['forward_lift'].min():.4f} - {w['forward_lift'].max():.4f}")
print(f"retained:     mean {w['retained'].mean():.1%} | "
      f"range {w['retained'].min():.1%} - {w['retained'].max():.1%}")
print(f"steps where the model beats random: {(w['forward_lift'] > 0).sum()} of {len(w)}")

2026-03-01: 36,272 pages | 31 clients | decline rate 0.3169


2026-03-31: 45,095 pages | 30 clients | decline rate 0.5131


2026-05-01: 42,079 pages | 32 clients | decline rate 0.6146


2026-06-01: 38,119 pages | 31 clients | decline rate 0.7869



     train       test  test_base_rate  gap_days  in_sample_lift  forward_lift  retained
2026-03-01 2026-03-31          0.5131        30          0.2287        0.2415    1.0556
2026-03-31 2026-05-01          0.6146        31          0.2377        0.1810    0.7618
2026-05-01 2026-06-01          0.7869        31          0.1818        0.1242    0.6832

forward lift: mean 0.1822 | range 0.1242 - 0.2415
retained:     mean 83.4% | range 68.3% - 105.6%
steps where the model beats random: 3 of 3


### 8b. Retention moved, so decompose it before believing it

The walk retains a mean **83.4%** where ML-08 reported **53%**, and the obvious reading — the model
decays with horizon, ML-08's gap was 61 days against these 30 — does not survive the numbers already
printed. **ML-08 scored 0.1291 on a pool declining at 0.7873. The last walk step scored 0.1242 on a
pool declining at 0.7869.** Same month, near-identical lift, one model 61 days stale and the other 31.
Staleness cannot be what separates 53% from 83.4%.

Retention is a ratio, and when a ratio moves the numerator and denominator have to be looked at
separately. `retained = forward_lift / in_sample_lift`, and the two runs differ almost entirely in the
**denominator**: ML-08's reference was earned at D1, this walk's third reference at 2026-05-01.

The test below pairs every lift by the **month it was scored on**, ignoring where the model came from,
and expresses each against the headroom that month allows. Ranking `k` pages out of a pool declining
at rate `b` cannot beat random by more than `1 − b`: at `b = 0.79` a perfect ranker gains at most
**0.21**, at `b = 0.51` it can gain **0.49**. If lift tracks headroom, the falling numbers are a
shrinking ceiling rather than a weakening model.

In [13]:
rows = []
for d in POINTS:
    br = coh[d]["declined"].mean()
    ins = w.loc[w["train"] == d, "in_sample_lift"]
    fwd = w.loc[w["test"] == d, "forward_lift"]
    rows.append({"scored_on": d, "base_rate": br, "headroom": 1 - br,
                 "in_sample": ins.iloc[0] if len(ins) else np.nan,
                 "forward": fwd.iloc[0] if len(fwd) else np.nan})
sm = pd.DataFrame(rows)
sm["fwd_vs_ins"] = sm["forward"] - sm["in_sample"]
sm["ins_pct_ceiling"] = sm["in_sample"] / sm["headroom"]
sm["fwd_pct_ceiling"] = sm["forward"] / sm["headroom"]

print("lift by the month it was SCORED on (model origin ignored)")
print(sm.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

both = sm.dropna(subset=["in_sample", "forward"])
print(f"\nmonths scored both ways: {len(both)} | "
      f"largest in-sample-to-forward gap {both['fwd_vs_ins'].abs().max():.4f}")

print("\nfraction of the available ceiling that was captured:")
cap = pd.concat([sm[["base_rate", "ins_pct_ceiling"]].rename(
                     columns={"ins_pct_ceiling": "pct_ceiling"}),
                 sm[["base_rate", "fwd_pct_ceiling"]].rename(
                     columns={"fwd_pct_ceiling": "pct_ceiling"})]).dropna()
print(f"  range {cap['pct_ceiling'].min():.1%} - {cap['pct_ceiling'].max():.1%}"
      f" | mean {cap['pct_ceiling'].mean():.1%}")

# ML-08's D2 point, recomputed here rather than quoted
ML08_D2_LIFT, ML08_D2_BASE, ML08_D1_LIFT = 0.1291, 0.7873, 0.2436
print(f"\nML-08 D2 (train 2026-03-31, test 2026-05-31, gap 61d):")
print(f"  lift {ML08_D2_LIFT:.4f} on base rate {ML08_D2_BASE:.4f}"
      f" -> {ML08_D2_LIFT / (1 - ML08_D2_BASE):.1%} of ceiling")
last = sm.iloc[-1]
print(f"  walk  {last['forward']:.4f} on base rate {last['base_rate']:.4f}"
      f" -> {last['fwd_pct_ceiling']:.1%} of ceiling  (gap 31d)")
print(f"  difference in lift between a 61-day-stale and a 31-day-stale model"
      f" on the same month: {abs(ML08_D2_LIFT - last['forward']):.4f}")
print(f"\nML-08 retention {ML08_D2_LIFT / ML08_D1_LIFT:.0%} uses a denominator earned at "
      f"base rate {sm.iloc[1]['base_rate']:.4f};")
print(f"  the walk's final step uses one earned at base rate {sm.iloc[2]['base_rate']:.4f}"
      f" ({sm.iloc[2]['in_sample']:.4f} vs {ML08_D1_LIFT:.4f}).")

lift by the month it was SCORED on (model origin ignored)
 scored_on  base_rate  headroom  in_sample  forward  fwd_vs_ins  ins_pct_ceiling  fwd_pct_ceiling
2026-03-01     0.3169    0.6831     0.2287      NaN         NaN           0.3348              NaN
2026-03-31     0.5131    0.4869     0.2377   0.2415      0.0038           0.4881           0.4959
2026-05-01     0.6146    0.3854     0.1818   0.1810     -0.0008           0.4717           0.4697
2026-06-01     0.7869    0.2131        NaN   0.1242         NaN              NaN           0.5829

months scored both ways: 2 | largest in-sample-to-forward gap 0.0038

fraction of the available ceiling that was captured:
  range 33.5% - 58.3% | mean 47.4%

ML-08 D2 (train 2026-03-31, test 2026-05-31, gap 61d):
  lift 0.1291 on base rate 0.7873 -> 60.7% of ceiling
  walk  0.1242 on base rate 0.7869 -> 58.3% of ceiling  (gap 31d)
  difference in lift between a 61-day-stale and a 31-day-stale model on the same month: 0.0049

ML-08 retention 53% u

### 8c. Isolating staleness: hold the test month fixed, move the training date

8b leaned on a comparison across notebooks — ML-08's 61-day-stale model against this walk's 31-day
one — and that comparison confounds three things at once. ML-08 fits **`FULL` (5 features)**; this walk
fits **`LEAN` (4)**, `FULL` minus `content_age_days`. ML-08 scores at 2026-05-31 and the walk at
2026-06-01. ML-08 fits on its whole evaluation pool, the walk on a `cohort_at` rebuild. A 0.0040 gap
across all that is suggestive, not clean.

**The clean version costs nothing**, because the cohorts are already built. Fix the test month, then
train the *same* pipeline at every earlier decision point. Everything except the age of the training
data is then held constant, and the only thing left to explain a change in lift is staleness.

In [14]:
fix = []
for test_d in POINTS[1:]:
    te = coh[test_d]
    rng_ = np.random.default_rng(0)
    p_rand = np.mean([queue_precision(te.assign(s=rng_.random(len(te))), "s", K, False)
                      for _ in range(20)])
    for train_d in POINTS:
        if train_d >= test_d:
            continue
        tr = coh[train_d]
        sc_ = StandardScaler().fit(tr[LEAN])
        m = Ridge(alpha=1.0).fit(sc_.transform(tr[LEAN]), tr["target"])
        p = queue_precision(te.assign(s=m.predict(sc_.transform(te[LEAN]))), "s", K, True)
        fix.append({"test": test_d, "base_rate": te["declined"].mean(), "train": train_d,
                    "gap_days": (pd.Timestamp(test_d) - pd.Timestamp(train_d)).days,
                    "lift": p - p_rand})

fx = pd.DataFrame(fix)
print("same test month, same pipeline, only the training date moves")
for test_d, g in fx.groupby("test"):
    g = g.sort_values("gap_days")
    span = g["lift"].max() - g["lift"].min()
    print(f"\n  test {test_d}  (base rate {g['base_rate'].iloc[0]:.4f}, "
          f"{len(g)} training date{'s' if len(g) > 1 else ''})")
    for _, r in g.iterrows():
        print(f"      trained {r['train']}  {r['gap_days']:>3.0f} days back   "
              f"lift {r['lift']:.4f}")
    if len(g) > 1:
        print(f"      spread across {int(g['gap_days'].min())}-{int(g['gap_days'].max())} "
              f"days of staleness: {span:.4f}")

multi = fx.groupby("test").filter(lambda g: len(g) > 1)
print(f"\nfreshest-minus-stalest, per test month:")
for test_d, g in multi.groupby("test"):
    g = g.sort_values("gap_days")
    d = g["lift"].iloc[0] - g["lift"].iloc[-1]
    print(f"  {test_d}: {d:+.4f}  "
          f"({int(g['gap_days'].iloc[0])}d vs {int(g['gap_days'].iloc[-1])}d)")
print(f"\ncorrelation between staleness and lift, within test month: "
      f"{multi.groupby('test')[['gap_days', 'lift']].corr().iloc[0::2, 1].mean():+.4f}")
print(f"every model beats random: {(fx['lift'] > 0).all()}  "
      f"({len(fx)} train/test pairs)")

same test month, same pipeline, only the training date moves

  test 2026-03-31  (base rate 0.5131, 1 training date)
      trained 2026-03-01   30 days back   lift 0.2415

  test 2026-05-01  (base rate 0.6146, 2 training dates)
      trained 2026-03-31   31 days back   lift 0.1810
      trained 2026-03-01   61 days back   lift 0.1616
      spread across 31-61 days of staleness: 0.0194

  test 2026-06-01  (base rate 0.7869, 3 training dates)
      trained 2026-05-01   31 days back   lift 0.1242
      trained 2026-03-31   62 days back   lift 0.1233
      trained 2026-03-01   92 days back   lift 0.1238
      spread across 31-92 days of staleness: 0.0009

freshest-minus-stalest, per test month:
  2026-05-01: +0.0194  (31d vs 61d)
  2026-06-01: +0.0004  (31d vs 92d)

correlation between staleness and lift, within test month: -0.7541
every model beats random: True  (6 train/test pairs)


**Verdict: what moves this model's advantage is the month it is pointed at, not how stale it is.
ML-08's "53% retained" is a ratio between two different ceilings.**

**Staleness, measured cleanly, is nearly free.** Fixing the test month and moving only the training
date — same pipeline, same features, same pool — gives:

| test month | base rate | 31 days back | 61-62 days back | 92 days back | spread |
|---|---|---|---|---|---|
| 2026-05-01 | 0.6146 | **0.1810** | **0.1616** | — | **0.0194** |
| 2026-06-01 | 0.7869 | **0.1242** | **0.1233** | **0.1238** | **0.0009** |

A model trained three months before the month it scores does the same work as one trained one month
before: **0.1238** against **0.1242**. **All 6 train/test pairs beat random.**

**Fresher is still directionally better, and the earlier reading of that was wrong.** Within test month
the correlation between staleness and lift is **-0.7541**, and the one month with a real gap prefers
the fresher model by **0.0194**. 8b reached for a cross-notebook comparison — ML-08's 61-day-stale
0.1291 against this walk's 31-day-stale 0.1242 — and read the older model as marginally *better*. That
comparison was confounded: ML-08 fits `FULL` (5 features) against this walk's `LEAN` (4), and the two
score a day apart off differently-built pools. **The controlled numbers are the ones to believe, and
they point the other way.** The conclusion survives because the effect is small either way, not
because the direction held.

**Set that against what the month is worth.** Lift runs **0.2415** at a base rate of 0.5131 down to
**0.1242** at 0.7869. The worst staleness penalty anywhere in this test is **0.0194**; changing which
month the model is scored on is worth several times that, and on 2026-06-01 the entire 31-to-92-day
span is worth **0.0009**.

**The month matters because the ceiling does.** A queue drawn from a pool declining at rate `b` cannot
beat random by more than `1 − b`. As the base rate climbs the room collapses — and the share of that
room the model captures does not:

| scored on | base rate | headroom | forward lift | share of ceiling |
|---|---|---|---|---|
| 2026-03-31 | 0.5131 | 0.4869 | 0.2415 | **0.4959** |
| 2026-05-01 | 0.6146 | 0.3854 | 0.1810 | **0.4697** |
| 2026-06-01 | 0.7869 | 0.2131 | 0.1242 | **0.5829** |

Forward lift halves; capture stays in a band and ends **higher** than it began. Across every
measurement it runs **33.5% - 58.3%**, mean **47.4%**.

**So the 53% is a ratio between two ceilings.** `retained = forward_lift / in_sample_lift`, and ML-08
divided a lift earned at base rate 0.7873 by one earned at **0.5131** — headroom 0.2131 against
0.4869. Less than half the room, hence roughly half the number. The walk's third step divides by a
reference earned at **0.6146** (**0.1818** against ML-08's **0.2436**) and reports **68.3%** for the
same model on the same month. **Neither figure describes decay; both describe which month the
denominator came from.**

**Retention above 100% makes the point from the other side.** The first step returns **105.6%** — the
model scored better on a month it had never seen than on the month it trained on, because the unseen
month was easier. A quantity that behaves like that is not measuring loss.

**The model beats random on 3 of 3 forward steps**, mean forward lift **0.1822**.

**What the walk found that no split could.** The decline rate moves **0.3169 → 0.5131 → 0.6146 →
0.7869** across four months. Every grouped split in this project — ten seeds, every stability range
quoted anywhere — resamples clients at D1 and therefore holds the base rate fixed at **0.5131**. Those
splits measure variation between clients; they are structurally blind to variation over time, and time
is the axis a deployed queue moves along. **The pool this model would be pointed at is not stationary,
and nothing here could previously have detected that.**

**Practical consequence.** Retraining cadence is not the lever — a 92-day-old model costs **0.0009**
here. Reporting is: the same model will post very different precision month to month for reasons that
have nothing to do with the model, so **an absolute `P@100` quoted without its base rate is not
interpretable**, and ML-10's playbook should carry lift over the month's own random bar instead.

**Stated limit.** Four decision points, six train/test pairs, one test month with three training
dates. Consecutive 90-day windows share 60 days and consecutive cohorts are largely the same pages, so
these are not independent folds and the staleness curve rests on two comparisons. Ninety-two days is
also the longest gap the warehouse permits — nothing here speaks to a model left running a year.

> ⚠️ **Superseded in design by FlyRank's answer (2026-08-19).** These decision points are spaced
> *monthly* because ML-03 had settled on a monthly cadence. Queues are rebuilt **weekly**, so the
> walk that matches deployment is weekly: the same 2026-03-01 to 2026-06-01 window holds about
> fourteen weekly decision points and thirteen steps, against four and three here.
>
> That is a materially better test than this one — the staleness curve currently rests on two
> comparisons and would rest on a dozen. Nothing above is wrong at monthly spacing; it is simply
> answering the cadence question the project assumed rather than the one it has. `cohort_at` takes an
> arbitrary date and `POINTS` is a list, so the rebuild is a change of spacing, not of code.
